# QuickDraw Sketch Recognition — Local Pipeline

This notebook orchestrates the same Day 1-5 pipeline that originally ran as
separate Kaggle notebooks/scripts, now adapted to run locally (this Mac,
Apple Silicon MPS backend). Each "Day" is still a standalone script
(`dayN_*.py`) — this notebook just runs them in order and displays results,
exactly like the original Kaggle orchestrator did.

**Before running anything below:**
1. Select the **"QuickDraw (.venv)"** kernel (Kernel picker, top right) — it has
   torch/torchvision/pandas/etc. already installed in `.venv`.
2. Configure your dataset in `local_config.py` (or via the
   `QUICKDRAW_KAGGLE_DATASET` env var) with your Kaggle dataset's
   `owner/dataset-slug`.
3. Make sure a Kaggle API token is available at `~/.kaggle/kaggle.json`
   (from https://www.kaggle.com/settings -> Create New Token), or set the
   `KAGGLE_USERNAME`/`KAGGLE_KEY` environment variables.

Day 6 (cross-dataset validation) and `quickdraw-baseline.ipynb` (a separate
TensorFlow/Keras experiment) are intentionally out of scope for this
notebook — see the note at the bottom.


In [ ]:
import sys
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS (Apple Silicon) available: {torch.backends.mps.is_available()}")


## Step 0: Get the dataset

Downloads (or reuses a cached copy of) the QuickDraw dataset from Kaggle into `./data/quickdraw_data`, using the identifier configured in `local_config.py`.

In [ ]:
from local_config import DATA_DIR
import os

if os.path.exists(DATA_DIR) and any(f.endswith('.npy') for f in os.listdir(DATA_DIR)):
    print(f"✓ Dataset already present at {DATA_DIR}, skipping download.")
else:
    get_ipython().system('{sys.executable} download_dataset.py')


## Step 0b: (Optional) Reset previous local outputs

Mirrors the original notebook's Kaggle working-directory cleanup. **Off by default** so it never silently deletes results you want to keep — flip `RESET_OUTPUT = True` to wipe `./output` before a fresh end-to-end run.

In [ ]:
import shutil
from local_config import OUTPUT_DIR

RESET_OUTPUT = False  # set True to wipe ./output before re-running everything

if RESET_OUTPUT and os.path.exists(OUTPUT_DIR):
    for filename in os.listdir(OUTPUT_DIR):
        file_path = os.path.join(OUTPUT_DIR, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            print(f'Failed to delete {file_path}. Reason: {e}')
    print(f"Cleared {OUTPUT_DIR}")
else:
    print("Skipping reset (RESET_OUTPUT is False).")


## Step 0c: List local dataset files

Local equivalent of walking `/kaggle/input`.

In [ ]:
from local_config import DATA_ROOT

for root, dirs, files in os.walk(DATA_ROOT):
    for file in files:
        print(os.path.join(root, file))


## Step 1: Verify the dataset

In [ ]:
get_ipython().system('{sys.executable} check_dataset.py')

## Day 1 — Dataset prep, EDA, ResNet18 baseline

In [ ]:
get_ipython().system('{sys.executable} day1_setup_baseline.py')

## Day 2 — Few-shot learning, augmentation, transfer learning experiments

In [ ]:
get_ipython().system('{sys.executable} day2_experiments.py')

## Day 3 — Results analysis and research paper generation

In [ ]:
get_ipython().system('{sys.executable} day3_analysis_and_paper.py')

## Day 4 — Alternative models comparison

In [ ]:
get_ipython().system('{sys.executable} day4_other_models_test.py')

## Display all plots from Day 1-4

In [ ]:
"""
Display All Plots from Day 1-4
Shows all generated visualizations inline in the notebook
"""

import os
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import numpy as np
from local_config import OUTPUT_DIR

# ============================
# CONFIGURATION
# ============================

RESULTS_DIRS = {
    'Day 1': os.path.join(OUTPUT_DIR, 'results/day1'),
    'Day 2': os.path.join(OUTPUT_DIR, 'results/day2'),
    'Day 3': os.path.join(OUTPUT_DIR, 'results/day3'),
    'Day 4': os.path.join(OUTPUT_DIR, 'results/day4')
}

print("="*80)
print("DISPLAYING ALL PLOTS FROM DAY 1-4")
print("="*80)

# ============================
# FUNCTION TO DISPLAY IMAGES
# ============================

def display_image(image_path, title=""):
    """Display a single image with title"""
    if os.path.exists(image_path):
        img = Image.open(image_path)
        plt.figure(figsize=(16, 10))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title, fontsize=16, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.show()
        print(f"✓ Displayed: {os.path.basename(image_path)}\n")
        return True
    else:
        print(f"❌ Not found: {os.path.basename(image_path)}\n")
        return False

def display_multiple_images(image_paths, titles, suptitle=""):
    """Display multiple images in a grid"""
    n_images = len(image_paths)
    if n_images == 0:
        return

    # Determine grid size
    if n_images <= 2:
        rows, cols = 1, n_images
        figsize = (16, 8)
    elif n_images <= 4:
        rows, cols = 2, 2
        figsize = (16, 16)
    elif n_images <= 6:
        rows, cols = 2, 3
        figsize = (18, 12)
    else:
        rows = (n_images + 2) // 3
        cols = 3
        figsize = (18, 6 * rows)

    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if n_images == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if n_images > 1 else [axes]

    for idx, (img_path, title) in enumerate(zip(image_paths, titles)):
        if os.path.exists(img_path):
            img = Image.open(img_path)
            axes[idx].imshow(img)
            axes[idx].axis('off')
            axes[idx].set_title(title, fontsize=12, fontweight='bold')
        else:
            axes[idx].text(0.5, 0.5, 'Image not found',
                          ha='center', va='center', fontsize=12)
            axes[idx].axis('off')

    # Hide unused subplots
    for idx in range(n_images, len(axes)):
        axes[idx].axis('off')

    if suptitle:
        fig.suptitle(suptitle, fontsize=18, fontweight='bold', y=0.995)

    plt.tight_layout()
    plt.show()
    print(f"✓ Displayed {n_images} images\n")

# ============================
# DAY 1: BASELINE & EDA
# ============================

print("\n" + "="*80)
print("DAY 1: BASELINE MODEL & EXPLORATORY DATA ANALYSIS")
print("="*80)

day1_dir = RESULTS_DIRS['Day 1']

if os.path.exists(day1_dir):
    day1_plots = [
        ('sample_images.png', 'Sample Sketches from Each Category'),
        ('class_distribution.png', 'Class Distribution (Balanced Dataset)'),
        ('training_curves.png', 'Training & Validation Curves'),
        ('confusion_matrix.png', 'Confusion Matrix - All 50 Categories')
    ]

    for plot_file, plot_title in day1_plots:
        plot_path = os.path.join(day1_dir, plot_file)
        display_image(plot_path, f"DAY 1: {plot_title}")
else:
    print(f"❌ Day 1 results directory not found: {day1_dir}\n")

# ============================
# DAY 2: EXPERIMENTS
# ============================

print("\n" + "="*80)
print("DAY 2: FEW-SHOT, AUGMENTATION & TRANSFER LEARNING")
print("="*80)

day2_dir = RESULTS_DIRS['Day 2']

if os.path.exists(day2_dir):
    day2_plots = [
        ('fewshot_results.png', 'Few-Shot Learning Results'),
        ('augmentation_results.png', 'Augmentation Strategy Comparison'),
        ('transfer_tsne.png', 't-SNE Visualization of Transfer Learning')
    ]

    for plot_file, plot_title in day2_plots:
        plot_path = os.path.join(day2_dir, plot_file)
        display_image(plot_path, f"DAY 2: {plot_title}")
else:
    print(f"❌ Day 2 results directory not found: {day2_dir}\n")

# ============================
# DAY 3: ANALYSIS & PAPER
# ============================

print("\n" + "="*80)
print("DAY 3: COMPREHENSIVE ANALYSIS & RESEARCH PAPER")
print("="*80)

day3_dir = RESULTS_DIRS['Day 3']

if os.path.exists(day3_dir):
    day3_plots = [
        ('overall_comparison.png', 'Overall Performance Comparison'),
        ('learning_curves.png', 'Learning Curves Analysis'),
        ('augmentation_improvement.png', 'Augmentation Improvement Analysis'),
        ('transfer_learning.png', 'Transfer Learning Summary')
    ]

    for plot_file, plot_title in day3_plots:
        plot_path = os.path.join(day3_dir, plot_file)
        display_image(plot_path, f"DAY 3: {plot_title}")
else:
    print(f"❌ Day 3 results directory not found: {day3_dir}\n")

# ============================
# DAY 4: MODEL COMPARISON
# ============================

print("\n" + "="*80)
print("DAY 4: ALTERNATIVE MODELS COMPARISON")
print("="*80)

day4_dir = RESULTS_DIRS['Day 4']

if os.path.exists(day4_dir):
    day4_plots = [
        ('model_comparison.png', 'Model Performance Comparison'),
        ('efficiency_comparison.png', 'Model Efficiency Analysis')
    ]

    for plot_file, plot_title in day4_plots:
        plot_path = os.path.join(day4_dir, plot_file)
        display_image(plot_path, f"DAY 4: {plot_title}")
else:
    print(f"❌ Day 4 results directory not found: {day4_dir}\n")

# ============================
# SUMMARY VIEW: ALL PLOTS TOGETHER
# ============================

print("\n" + "="*80)
print("SUMMARY: ALL PLOTS OVERVIEW")
print("="*80)

all_plots = []
all_titles = []

for day_name, day_dir in RESULTS_DIRS.items():
    if os.path.exists(day_dir):
        png_files = sorted(glob.glob(os.path.join(day_dir, '*.png')))
        for png_file in png_files:
            all_plots.append(png_file)
            all_titles.append(f"{day_name}: {os.path.basename(png_file).replace('.png', '').replace('_', ' ').title()}")

if all_plots:
    print(f"\nFound {len(all_plots)} total plots across all days")
    print("\nDisplaying all plots in grid view...\n")
    display_multiple_images(all_plots, all_titles, "Complete Project Visualization Gallery")
else:
    print("❌ No plots found\n")

# ============================
# STATISTICS SUMMARY
# ============================

print("\n" + "="*80)
print("VISUALIZATION STATISTICS")
print("="*80)

for day_name, day_dir in RESULTS_DIRS.items():
    if os.path.exists(day_dir):
        png_files = glob.glob(os.path.join(day_dir, '*.png'))
        csv_files = glob.glob(os.path.join(day_dir, '*.csv'))
        txt_files = glob.glob(os.path.join(day_dir, '*.txt'))

        print(f"\n{day_name}:")
        print(f"   📊 PNG files: {len(png_files)}")
        print(f"   📄 CSV files: {len(csv_files)}")
        print(f"   📝 TXT files: {len(txt_files)}")

        if png_files:
            print(f"   Plots:")
            for png_file in sorted(png_files):
                print(f"      - {os.path.basename(png_file)}")

# ============================
# COMPLETION
# ============================

print("\n" + "="*80)
print("✅ ALL PLOTS DISPLAYED SUCCESSFULLY!")
print("="*80)

print("\n📊 Total Visualizations:")
print(f"   Day 1: {len(glob.glob(os.path.join(RESULTS_DIRS['Day 1'], '*.png'))) if os.path.exists(RESULTS_DIRS['Day 1']) else 0} plots")
print(f"   Day 2: {len(glob.glob(os.path.join(RESULTS_DIRS['Day 2'], '*.png'))) if os.path.exists(RESULTS_DIRS['Day 2']) else 0} plots")
print(f"   Day 3: {len(glob.glob(os.path.join(RESULTS_DIRS['Day 3'], '*.png'))) if os.path.exists(RESULTS_DIRS['Day 3']) else 0} plots")
print(f"   Day 4: {len(glob.glob(os.path.join(RESULTS_DIRS['Day 4'], '*.png'))) if os.path.exists(RESULTS_DIRS['Day 4']) else 0} plots")

total_plots = sum([
    len(glob.glob(os.path.join(day_dir, '*.png')))
    for day_dir in RESULTS_DIRS.values()
    if os.path.exists(day_dir)
])

print(f"\n🎨 Total: {total_plots} visualizations across all days")
print("\n" + "="*80)


## ResNet18 parameter analysis

In [ ]:
#Parameter Count

# Run this in your notebook to get exact numbers:

import torch
from torchvision import models
import torch.nn as nn

# Create your ResNet18 model (same as Day 1)
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 50)  # 50 classes for your dataset

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print("="*60)
print("RESNET18 PARAMETER ANALYSIS")
print("="*60)
print(f"\nTotal Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Frozen Parameters: {frozen_params:,}")
print(f"\nMemory Usage:")
print(f"  Model Size: {total_params * 4 / (1024**2):.2f} MB (float32)")
print(f"  Training Memory (estimated): {total_params * 4 * 2 / (1024**2):.2f} MB")

# Layer-wise breakdown
print("\n" + "="*60)
print("LAYER-WISE PARAMETER COUNT")
print("="*60)

for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name:40s} {param.numel():>12,} params")


## Day 5 — Custom efficient SketchNet model

In [ ]:
get_ipython().system('{sys.executable} day5_custom_efficient_model.py')

## Out of scope (for now): Day 6 & quickdraw-baseline.ipynb

- **`day6_cross_dataset_validation.py`** — cross-dataset validation of the Day 5
  SketchNet model against a second dataset (ImageNet-Sketch / handdrawn shapes).
  Still uses Kaggle-style paths; wire it up the same way as the scripts above
  (add its dataset to `local_config.py`, swap `/kaggle/...` paths) once you're
  ready to run it.
- **`quickdraw-baseline.ipynb`** — an independent TensorFlow/Keras experiment
  (MobileNetV3, ResNet50, EfficientNetB0, SwinTiny + CBAM attention) on the
  same QuickDraw data. Not part of this PyTorch pipeline; left untouched.
